# Exact Inference vs Mean Field vs Gibbs Sampling

This notebook presents a cleaned version of the **Mean Field Approximation and Gibbs Sampling** project from a graphical models coursework portfolio.

## Goal
Compare:
1. exact inference via transfer-matrix / column clustering
2. fully factorized mean field approximation
3. checkerboard Gibbs sampling

on a binary lattice model under different coupling strengths.

## What this notebook demonstrates
- exact versus approximate inference
- symmetry breaking in mean field
- slow mixing and mode trapping in Gibbs sampling
- how inference quality changes with coupling strength

## Implementation

The code below computes corner marginals under several values of `beta` and reports results for:
- Exact inference
- Mean Field (raw and symmetrized)
- Gibbs sampling (single-chain and multi-chain)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
from scipy.special import logsumexp

from scipy.special import logsumexp
# 1. Exact Inference (Transfer Matrix)
def solve_exact(beta, N=10):
    """Exact inference O(N * 4^N). Computes P(top, bot) and log Z."""
    dim = 2**N
    states = np.array([list(np.binary_repr(i, width=N)) for i in range(dim)], dtype=int)

    # Potentials
    matches_v = (states[:, :-1] == states[:, 1:]).sum(axis=1)
    log_V = matches_v * beta
    matches_h = (states[:, None, :] == states[None, :, :]).sum(axis=2)
    log_M = matches_h * beta

    # Forward Pass
    log_alpha = log_V.copy()
    for _ in range(N - 1):
        term = log_alpha[:, None] + log_M
        msg = logsumexp(term, axis=0)
        log_alpha = msg + log_V

    log_Z = logsumexp(log_alpha)
    p_last = np.exp(log_alpha - log_Z)

    # Marginalize to corners
    joint = np.zeros((2, 2))
    for i in range(dim):
        top = states[i, 0]
        bot = states[i, -1]
        joint[top, bot] += p_last[i]

    return joint, log_Z

# 2. Mean Field (Returns Raw & Symmetrized)
def solve_mean_field_comprehensive(beta, N=10, max_iter=1000):
    """Run MF twice (biased 0 and biased 1) to show breaking vs symmetrization."""
    results = []

    # Run 1: Bias towards 0
    # Run 2: Bias towards 1
    for init_bias in [0.2, 0.8]:
        mu = np.random.rand(N, N) * 0.1 + (init_bias - 0.05)
        for _ in range(max_iter):
            mu_old = mu.copy()
            for r in range(N):
                for c in range(N):
                    neigh_sum = 0.0
                    for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
                        rr, cc = r+dr, c+dc
                        if 0 <= rr < N and 0 <= cc < N:
                            neigh_sum += (2 * mu[rr, cc] - 1)
                    log_odds = beta * neigh_sum
                    mu[r, c] = 1.0 / (1.0 + np.exp(-log_odds))
            if np.max(np.abs(mu - mu_old)) < 1e-6: break

        mt, mb = mu[0, N-1], mu[N-1, N-1]
        results.append(np.outer([1-mt, mt], [1-mb, mb]))

    # Return Raw (from first run) and Symmetrized (average of both)
    return results[0], 0.5 * (results[0] + results[1])
# 3. Gibbs Sampling (Returns Single & Multi)
def solve_gibbs_comprehensive(beta, N=10, samples=5000, burn_in=1000):
    """Compare Single Chain (All-0 init) vs Multi-Chain (All-0 + All-1)."""
    chains = [np.zeros((N,N), dtype=int), np.ones((N,N), dtype=int)]
    chain_counts = []

    # Checkerboard masks
    grid_indices = np.indices((N,N))
    mask_black = (grid_indices[0] + grid_indices[1]) % 2 == 0
    mask_white = ~mask_black

    for grid in chains:
        counts = np.zeros((2, 2))
        for i in range(burn_in + samples):
            for mask in [mask_black, mask_white]:
                rows, cols = np.where(mask)
                for r, c in zip(rows, cols):
                    n0, n1 = 0, 0
                    for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
                        rr, cc = r+dr, c+dc
                        if 0 <= rr < N and 0 <= cc < N:
                            if grid[rr, cc] == 0: n0 += 1
                            else: n1 += 1
                    log_odds = beta * (n1 - n0)
                    if log_odds > 50: p1 = 1.0
                    elif log_odds < -50: p1 = 0.0
                    else: p1 = 1.0 / (1.0 + np.exp(-log_odds))
                    grid[r, c] = 1 if np.random.rand() < p1 else 0

            if i >= burn_in:
                counts[grid[0, N-1], grid[N-1, N-1]] += 1
        chain_counts.append(counts / np.sum(counts))

    # Return Single Chain (Chain 0) and Multi-Chain (Average)
    return chain_counts[0], 0.5 * (chain_counts[0] + chain_counts[1])

# Execution
def run_final_part3():
    betas = [4, 1, 0.01]
    _, logz_1 = solve_exact(1.0)
    print(f"=== Part 3 Results (LogZ for beta=1: {logz_1:.4f}) ===\n")

    print(f"{'Beta':<5} | {'Method':<20} | {'P(0,0)':<8} | {'P(1,1)':<8} | {'Note'}")
    print("-" * 85)

    for b in betas:
        # Exact
        p_ex, _ = solve_exact(b)
        print(f"{b:<5} | {'Exact':<20} | {p_ex[0,0]:.4f}   | {p_ex[1,1]:.4f}   | Benchmark")

        # Mean Field
        p_mf_raw, p_mf_sym = solve_mean_field_comprehensive(b, max_iter=2000 if b==4 else 1000)
        print(f"{b:<5} | {'MF (Raw)':<20} | {p_mf_raw[0,0]:.4f}   | {p_mf_raw[1,1]:.4f}   | {'Broken' if b==4 else ''}")
        print(f"{b:<5} | {'MF (Symmetrized)':<20} | {p_mf_sym[0,0]:.4f}   | {p_mf_sym[1,1]:.4f}   | Restored")

        # Gibbs
        p_gb_sng, p_gb_mul = solve_gibbs_comprehensive(b)
        print(f"{b:<5} | {'Gibbs (Single)':<20} | {p_gb_sng[0,0]:.4f}   | {p_gb_sng[1,1]:.4f}   | {'Stuck' if b==4 else ''}")
        print(f"{b:<5} | {'Gibbs (Multi)':<20} | {p_gb_mul[0,0]:.4f}   | {p_gb_mul[1,1]:.4f}   | Fixed")
        print("-" * 85)

if __name__ == "__main__":
    run_final_part3()